In [1]:
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image
import torch
from pathlib import Path
import sklearn
from tqdm import tqdm
import PIL
import pandas as pd
import numpy as np

#from smolvlm.translate import translate_en_to_ru # использовал для перевода в локальном тесте
from smolvlm.imagetotext import image_to_text_pipeline

MODEL_PATH = r"smolvlm/smolvlm-256m"   # локальная папка с моделью

C:\Users\User\anaconda3\envs\RESEARCHPY312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используем устройство: cpu


In [2]:
# # works only with old version of transformers
# !pip install -U transformers==4.37.1

In [3]:
# # there is a bug in ipykernel that prevents downloading of BAAI/bge-large-en-v1.5, so we need to be sure that it is updated
# # https://github.com/huggingface/xet-core/issues/526
# !pip install -U ipykernel>=7.1.0

# Calculate Metrics for UForm Model

In [4]:
def get_metrics_txt_emb(text):

    """
    Generates a normalized text embedding using BGE-large-en-v1.5 model.
    
    Args:
        text (str): Input text to generate embedding for
        
    Returns:
        numpy.ndarray: Normalized embedding vector (shape: [1024])
    """
    
    METRICS_TXT_EMB_MODEL_NAME = r"smolvlm/bge"

    tokenizer = transformers.AutoTokenizer.from_pretrained(METRICS_TXT_EMB_MODEL_NAME)
    model = transformers.AutoModel.from_pretrained(METRICS_TXT_EMB_MODEL_NAME)
    model.eval()
    
    encoded_input = tokenizer([text], padding=True, truncation=True, return_tensors='pt')
    
    with torch.no_grad():
        model_output = model(**encoded_input)
        sentence_embeddings = model_output[0][:, 0]

    return torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)[0].numpy()

## Load Data

In [5]:
def load_image_markup(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        while True:
            line = f.readline()
            if len(line) == 0:
                break
            sep_ind = line.find("\"")
            path = ("../" + line[:sep_ind]).strip()
            desc = line[sep_ind + 1:].strip()[:-1]
            data.append((path, desc))
    return data


img_markup = load_image_markup("../data/matching_images.txt")

## Metrics

In [6]:
def calc_metrics(markup, model, processor_obj):

    """
    Calculates VLM retrieval metrics using distance-based ranking and optimal F1 threshold.
    
    Args:
        markup (list): List of [image_path, text_description] pairs
        model: Vision-Language Model for text generation
        processor_obj: Model processor/tokenizer
        
    Returns:
        tuple: (tp, fp, tn, fn, opt_thr, tp_list, fp_list)
    """

    PROMPT = "Describe the product in the image for an online sales listing. First, write a 2–3 sentence description. Then add a line starting with 'Keywords:' followed by 8–15 comma-separated keywords about type, color, material, style, use cases, and target audience."
    # это актуально для модели smolvlm
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"{PROMPT}"},
        ],
    }]
    
    text_list = [markup_line[1].lower() for markup_line in markup]
    # some descriptions are the same or mostly same (start from the same words)
    unique_indices = []
    for idx in range(len(text_list)):
        if idx == 0:
            unique_indices.append(idx)
            continue
        prev_started = any(txt.startswith(text_list[idx]) for txt in text_list[:idx])
        current_started = any(text_list[idx].startswith(txt) for txt in text_list[:idx])
        if prev_started or current_started:
            continue
        unique_indices.append(idx)
    unique_text_list = [text_list[idx] for idx in unique_indices]
    emb_unique_text_list = [get_metrics_txt_emb(txt) for txt in unique_text_list]

    y_true_list = []
    distance_list = []
    for img_idx, markup_line in tqdm(enumerate(markup), total=len(text_list)):
        
        # это кастонаия функция для генерации тектса от модели
        decoded_out = image_to_text_pipeline(model_path = MODEL_PATH , image_path = markup_line[0], message=messages)

        emb_decoded_out = get_metrics_txt_emb(decoded_out[0].split('Assistant: ')[1]) #decoded_out[0].split('Assistant: ')[1] - это специфика ответа модели
        distance_per_image = [np.linalg.norm(emb_unique_text - emb_decoded_out) for emb_unique_text in emb_unique_text_list]

        if img_idx not in unique_indices:
            assert markup_line[1].lower() == text_list[img_idx]
            txt_to_find = markup_line[1].lower()
            u_img_idx = -1
            for i in range(len(unique_text_list)):
                if unique_text_list[i].startswith(txt_to_find) or txt_to_find.startswith(unique_text_list[i]):
                    u_img_idx = i
                    break
            assert u_img_idx >= 0
            assert u_img_idx < img_idx
        else:
            u_img_idx = unique_indices.index(img_idx)

        y_true_list += [1 if idx == u_img_idx else 0 for idx in range(len(unique_text_list))]
        distance_list += distance_per_image

    y_true_vec = np.array(y_true_list)
    max_distance = max(distance_list)
    logit_vec = np.array([(max_distance - dist)/max_distance for dist in distance_list])
    assert y_true_vec.shape == logit_vec.shape

    assert logit_vec.min() >= 0
    assert logit_vec.max() <= 1
    assert np.allclose(np.sort(np.unique(y_true_vec)), np.array([0, 1]))

    # https://stats.stackexchange.com/q/287117/
    prevalence = np.count_nonzero(y_true_vec == 1, keepdims=False)/len(y_true_vec)
    assert prevalence > 0
    fpr_vec, tpr_vec, thr_vec = sklearn.metrics.roc_curve(y_true_vec, logit_vec)
    recall_vec = tpr_vec
    tnr_vec = 1 - fpr_vec

    zero_div_idxs = np.where((recall_vec*prevalence) + ((1 - tnr_vec)*(1 - prevalence)) == 0)[0]
    if zero_div_idxs.size > 0:
        # to avoid zero-division warning from numpy below
        recall_vec[zero_div_idxs] += 1e-8
    precision_vec = (recall_vec*prevalence)/((recall_vec*prevalence) + ((1 - tnr_vec)*(1 - prevalence)))

    zero_div_idxs = np.where(precision_vec + recall_vec == 0)[0]
    if zero_div_idxs.size > 0:
        # to avoid zero-division warning from numpy below
        precision_vec[zero_div_idxs] += 1e-8
        recall_vec[zero_div_idxs] += 1e-8
    f1_vec = 2*(precision_vec*recall_vec)/(precision_vec + recall_vec)

    opt_thr = thr_vec[np.argmax(f1_vec)]
    if np.isinf(opt_thr):
        opt_thr = 1

    # using ">=" below instead of ">" is extremely important, because sklearn cn return edge values for threshold
    tp = np.count_nonzero((logit_vec >= opt_thr) & (y_true_vec == 1))
    fp = np.count_nonzero((logit_vec >= opt_thr) & (y_true_vec == 0))
    tn = np.count_nonzero((logit_vec < opt_thr) & (y_true_vec == 0))
    fn = np.count_nonzero((logit_vec < opt_thr) & (y_true_vec == 1))

    tp_list = [
        (distance_list[tp_idx], text_list[tp_idx % len(unique_text_list)])
        for tp_idx in np.nonzero((logit_vec >= opt_thr) & (y_true_vec == 1))[0]
    ]
    tp_list = [(sum(dist for dist, txt in tp_list if txt == u_txt), u_txt) for u_txt in set([txt for dist, txt in tp_list])]
    tp_list.sort(key=lambda item: item[0])
    fp_list = [
        (distance_list[fp_idx], text_list[fp_idx % len(unique_text_list)])
        for fp_idx in np.nonzero((logit_vec >= opt_thr) & (y_true_vec == 0))[0]
    ]
    fp_list = [(sum(dist for dist, txt in fp_list if txt == u_txt), u_txt) for u_txt in set([txt for dist, txt in fp_list])]
    fp_list.sort(key=lambda item: item[0])
    opt_thr = max_distance - (float(opt_thr)*max_distance)

    return tp, fp, tn, fn, opt_thr, tp_list, fp_list

In [7]:
# MODEL_NAME = "unum-cloud/uform-gen2-qwen-500m"
# model = transformers.AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
# processor_func = transformers.AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
# это все в отдельном модуле

In [8]:
tp, fp, tn, fn, thr, tp_list, fp_list = calc_metrics(img_markup, _, _)

if 2*tp + fp + fn > 0:
    f1 = 2*tp/(2*tp + fp + fn)
else:
    f1 = 0

if tp + fp + tn + fn > 0:
    acc = (tp + tn)/(tp + fp + tn + fn)
else:
    acc = 0

print("TRUE POSITIVES:\n\t" + "\n\t".join(f"({dist:.1f}) {txt}" for dist, txt in tp_list) + "\n\n")
print("FALSE POSITIVES:\n\t" + "\n\t".join(f"({dist:.1f}) {txt}" for dist, txt in fp_list) + "\n\n")

  0%|                                                                                          | 0/311 [00:00<?, ?it/s]

Answer: **Image Description:**

The image displays two jackets laid out on a marble surface. The jackets are of different colors and materials. The left jacket is a light beige color, while the right


  0%|▎                                                                               | 1/311 [00:57<4:56:11, 57.33s/it]

Answer: The image shows a collection of fabric items laid out on a surface. The fabric appears to be made of a blend of various colors and materials. The topmost fabric is a light beige or off


  1%|▌                                                                               | 2/311 [01:57<5:03:19, 58.90s/it]

...

Answer: The image shows a pair of black shoes, specifically a pair of 10-centimeter heels. The shoes are made of a smooth, flexible material that appears to be made of a material that


100%|████████████████████████████████████████████████████████████████████████████▊| 310/311 [14:10:06<00:57, 57.55s/it]

Answer: The image depicts a pair of black Adidas shoes, specifically the Adidas 100, which is a high-top sneaker. The shoes are displayed on a platform, and the l


100%|████████████████████████████████████████████████████████████████████████████| 311/311 [14:11:01<00:00, 164.18s/it]

TRUE POSITIVES:
	(0.6) crib bumpers with bunnies
	(0.6) refrigerator, wooden kitchen table, stool
	(0.6) children's bathing chair
	(0.6) dark blue women's dress
	(0.6) porcelain teapots, white with floral print and black, sugar bowls, tea cups and saucers
	(0.6) black trousers and a black vest
	(0.6) textbooks for schoolchildren to prepare for exams
	(0.6) baby's ankle boots
	(0.7) warm zipped jacket with a hood
	(0.7) long red women's dress
	(0.7) gray knitted women's dress
	(0.7) girls clothing set
	(0.7) dark gray rabbit
	(0.7) women's warm down jacket with zip and hood
	(0.7) women's denim dress with long sleeves
	(0.7) women's fur coat
	(0.7) red baby blanket on a white chest of drawers
	(0.7) set of black and grey pants
	(0.7) black kitten sits on the wooden boards
	(0.7) pink women's blouse with short sleeves
	(0.7) abstract patterned floor rug
	(0.7) leather women's boots soles view
	(0.7) reusable diapers
	(0.7) inflatable swimming ring clasp
	(0.7) black and grey kittens are 

In [10]:
print(f"Model: smolvlm-256m")
print(f"TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")
print(f"Threshold: {thr}")
print(f"Accuracy: {acc}")
print(f"F1: {f1}")

Model: smolvlm-256m
TP: 62, FP: 239, TN: 75956, FN: 249
Threshold: 0.7467235326766968
Accuracy: 0.9936214153138316
F1: 0.20261437908496732
